# MAIRA-2 API on Kaggle (2× T4 + ngrok)

Run every cell from top to bottom in a fresh Kaggle session. Enable **GPU T4 ×2** and **Internet**, accept the `microsoft/maira-2` license, and add Kaggle secrets named `HF_TOKEN` and `NGROK_AUTHTOKEN`.

> MAIRA-2 is a research model and is not intended for standalone clinical use.

In [ ]:
# 1. Install the Transformers release officially tested with MAIRA-2.
# This cell automatically restarts once if Kaggle has Transformers 5.x loaded.
import importlib.metadata
import os
import subprocess
import sys

REQUIRED_TRANSFORMERS = "4.51.3"
try:
    installed_transformers = importlib.metadata.version("transformers")
except importlib.metadata.PackageNotFoundError:
    installed_transformers = None

packages_changed = installed_transformers != REQUIRED_TRANSFORMERS
if installed_transformers != REQUIRED_TRANSFORMERS:
    print(f"Installing dependencies (found transformers {installed_transformers})...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "--upgrade",
        "accelerate", "sentencepiece", "protobuf", "fastapi",
        "uvicorn[standard]", "python-multipart", "pyngrok", "nest_asyncio",
    ])
    # Pin Transformers last so another dependency cannot upgrade it to 5.x.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "--upgrade", "--force-reinstall",
        f"transformers=={REQUIRED_TRANSFORMERS}",
    ])
    importlib.metadata.distributions.cache_clear() if hasattr(importlib.metadata.distributions, "cache_clear") else None
    installed_transformers = importlib.metadata.version("transformers")
    assert installed_transformers == REQUIRED_TRANSFORMERS, (
        f"Installation failed: found transformers {installed_transformers}"
    )

loaded_transformers = sys.modules.get("transformers")
loaded_version = getattr(loaded_transformers, "__version__", None)
needs_restart = (
    packages_changed
    or (loaded_transformers is not None and loaded_version != REQUIRED_TRANSFORMERS)
)
if needs_restart:
    print("Dependencies installed. Restarting the Kaggle kernel now...", flush=True)
    os.kill(os.getpid(), 9)

print(f"Compatible Transformers distribution ready: {REQUIRED_TRANSFORMERS}")


In [ ]:
# 2. Validate the runtime and load Kaggle secrets.
# If this reports 5.x, run cell 1 and wait for Kaggle to reconnect first.
import transformers
import torch

assert transformers.__version__ == "4.51.3", (
    f"Expected transformers 4.51.3, found {transformers.__version__}. "
    "Restart the Kaggle session and run the notebook again."
)
assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle settings."
assert torch.cuda.device_count() >= 2, "Select the T4 ×2 accelerator."

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
NGROK_AUTHTOKEN = secrets.get_secret("NGROK_AUTHTOKEN")
assert HF_TOKEN, "Missing Kaggle secret: HF_TOKEN"
assert NGROK_AUTHTOKEN, "Missing Kaggle secret: NGROK_AUTHTOKEN"
login(token=HF_TOKEN, add_to_git_credential=False)

print("Transformers:", transformers.__version__)
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {torch.cuda.get_device_name(index)} ({props.total_memory / 2**30:.1f} GiB)")


In [ ]:
# 3. Load MAIRA-2 across both T4 GPUs. This can take several minutes.
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = "microsoft/maira-2"
processor = AutoProcessor.from_pretrained(
    MODEL_ID, trust_remote_code=True, token=HF_TOKEN
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

# The image tower and token embeddings are placed on GPU 0 by device_map=auto.
INPUT_DEVICE = torch.device("cuda:0")
print("MAIRA-2 loaded successfully.")
print(model.hf_device_map)


In [ ]:
# 4. Define the FastAPI service.
import asyncio
import io
import traceback
from typing import Optional

from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from PIL import Image, UnidentifiedImageError

app = FastAPI(title="MAIRA-2 Inference API")
generation_lock = asyncio.Lock()

def json_safe(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, tuple):
        return [json_safe(item) for item in value]
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    return value

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": MODEL_ID,
        "transformers": transformers.__version__,
        "cuda_devices": torch.cuda.device_count(),
    }

@app.post("/generate")
async def generate(
    frontal: UploadFile = File(..., description="Frontal chest X-ray image"),
    lateral: Optional[UploadFile] = File(None, description="Optional lateral image"),
    indication: str = Form("Indication not provided."),
    technique: str = Form("Single frontal chest radiograph."),
    comparison: str = Form("No prior study provided."),
    grounded: bool = Form(False),
):
    async with generation_lock:
        try:
            frontal_bytes = await frontal.read()
            if not frontal_bytes:
                raise ValueError("The frontal image is empty.")
            frontal_img = Image.open(io.BytesIO(frontal_bytes)).convert("RGB")

            lateral_img = None
            if lateral is not None:
                lateral_bytes = await lateral.read()
                if lateral_bytes:
                    lateral_img = Image.open(io.BytesIO(lateral_bytes)).convert("RGB")

            processed = processor.format_and_preprocess_reporting_input(
                current_frontal=frontal_img,
                current_lateral=lateral_img,
                prior_frontal=None,
                indication=indication,
                technique=technique,
                comparison=comparison,
                prior_report=None,
                return_tensors="pt",
                get_grounding=grounded,
            ).to(INPUT_DEVICE)

            with torch.inference_mode():
                output_ids = model.generate(
                    **processed,
                    max_new_tokens=450 if grounded else 300,
                    use_cache=True,
                )

            prompt_length = processed["input_ids"].shape[-1]
            decoded = processor.decode(
                output_ids[0][prompt_length:], skip_special_tokens=True
            ).lstrip()
            prediction = processor.convert_output_to_plaintext_or_grounded_sequence(decoded)

            if isinstance(prediction, tuple):
                report_text = prediction[0]
                boxes = prediction[1] if len(prediction) > 1 else []
            else:
                report_text = prediction
                boxes = []

            return {"report": str(report_text), "boxes": json_safe(boxes)}

        except (UnidentifiedImageError, ValueError) as exc:
            raise HTTPException(status_code=400, detail=str(exc)) from exc
        except torch.cuda.OutOfMemoryError as exc:
            traceback.print_exc()
            torch.cuda.empty_cache()
            raise HTTPException(status_code=503, detail=f"CUDA out of memory: {exc}") from exc
        except Exception as exc:
            traceback.print_exc()
            raise HTTPException(
                status_code=500, detail=f"{type(exc).__name__}: {exc}"
            ) from exc

print("FastAPI application configured.")


In [ ]:
# 5. Start Uvicorn in the background. Run this cell only once per session.
import threading
import time
import nest_asyncio
import uvicorn

nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(3)
assert server_thread.is_alive(), "Uvicorn failed to start."
print("Server started on port 8000.")


In [ ]:
# 6. Open the ngrok tunnel. The URL changes after a fresh Kaggle session.
from pyngrok import conf, ngrok

conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()
tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url.rstrip("/")

print(f"MAIRA_API_URL={public_url}")
print(f"Health:   {public_url}/health")
print(f"Generate: POST {public_url}/generate")
print("Update MAIRA_API_URL in MedoraAI/.env when this URL changes.")


In [ ]:
# 7. Verify the public service.
import requests

response = requests.get(
    f"{public_url}/health",
    headers={"ngrok-skip-browser-warning": "true"},
    timeout=30,
)
response.raise_for_status()
print("Status:", response.status_code)
print("Response:", response.json())


## Usage

MedoraAI sends `POST /generate` with multipart field `frontal`. The endpoint returns `{"report": "...", "boxes": []}`.

When Kaggle restarts, copy the newly printed `MAIRA_API_URL` into the project `.env` and restart the MedoraAI backend. Keep this Kaggle session running while using the endpoint.